# VQG 2016 Reproduction — Colab Pro+ training

Thin glue notebook: everything real lives in the repo's `scripts/` and `vqg/` as versioned `.py` files, so it's easy to debug locally too. This notebook just clones the repo, mounts Drive for persistent storage, and calls the scripts in order.

Runtime → Change runtime type → GPU (Colab Pro+ gives you a faster GPU + longer session limits) before running.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = "https://github.com/malimustafaa/vqg-Natural-Question-Generation.git"

REPO_DIR = "/content/vqg-2016-reproduction"
# Persistent storage on Drive -- images/features/checkpoints survive across sessions,
# so you don't re-download ~11k images or re-run VGG16 every time you reconnect.
DATA_DIR = "/content/drive/MyDrive/vqg-2016-reproduction-data"

import os
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

In [ ]:
%cd $REPO_DIR
!pip install -q -r requirements.txt

In [ ]:
# One-time: seed the Drive data dir with the small versioned CSVs from the repo.
# -n (no-clobber) so re-running this cell on a later day never overwrites
# manifest/vocab files that later steps have already built on Drive.
!mkdir -p $DATA_DIR/raw $DATA_DIR/images $DATA_DIR/features $DATA_DIR/checkpoints $DATA_DIR/results
!cp -n $REPO_DIR/data/raw/*.csv $DATA_DIR/raw/
!cp -n $REPO_DIR/data/raw/vocab.json $DATA_DIR/raw/ 2>/dev/null || true

## 1. Full-scale image download (into Drive)

Resumable -- safe to re-run if the session disconnects partway through. Expect roughly: coco ~4,990, flickr ~4,135, bing ~1,926 usable images (see README “Known deviations” for why bing is smaller than the paper's 5,000).

In [ ]:
!python scripts/download_images.py --dir $DATA_DIR --workers 32

## 2. Rebuild the vocab from the full training manifest (optional but recommended)

The `vocab.json` copied from the repo was built on the local dev sample. Once the full dataset is downloaded, rebuild it on the real, full `all_train.csv` questions so nothing is missing from the vocabulary.

In [ ]:
!python scripts/build_vocab.py --train-csv $DATA_DIR/raw/all_train.csv --out $DATA_DIR/raw/vocab.json

## 3. Extract frozen VGG16 fc7 features (GPU-accelerated here, unlike the local CPU smoke test)

In [ ]:
for split in ["train", "val", "test"]:
    !python scripts/extract_features.py \
        --manifest $DATA_DIR/raw/manifest_{split}.csv \
        --out $DATA_DIR/features/{split}.pt \
        --batch-size 64

## 4. Train GRNN_all (pooled coco+flickr+bing) and each GRNN_X (per-source)

Four training runs total, matching the paper's Table 5. Tune `--lr`/`--batch-size`/`--epochs` against
the printed validation loss -- the paper doesn't publish these hyperparameters (see README).

In [ ]:
RUNS = {
    "grnn_all": ["bing", "coco", "flickr"],
    "grnn_bing": ["bing"],
    "grnn_coco": ["coco"],
    "grnn_flickr": ["flickr"],
}

for name, sources in RUNS.items():
    sources_arg = " ".join(sources)
    !python scripts/train.py --sources {sources_arg} \
        --train-manifest $DATA_DIR/raw/manifest_train.csv --train-features $DATA_DIR/features/train.pt \
        --val-manifest $DATA_DIR/raw/manifest_val.csv --val-features $DATA_DIR/features/val.pt \
        --vocab $DATA_DIR/raw/vocab.json \
        --checkpoint-dir $DATA_DIR/checkpoints/{name} \
        --epochs 30 --batch-size 32 --lr 0.1 --patience 5

## 5. Decode (beam search, width 8) + evaluate (BLEU/METEOR) on the test split

In [ ]:
for name in RUNS:
    !python scripts/decode.py \
        --checkpoint $DATA_DIR/checkpoints/{name}/best.pt \
        --manifest $DATA_DIR/raw/manifest_test.csv --features $DATA_DIR/features/test.pt \
        --vocab $DATA_DIR/raw/vocab.json \
        --out $DATA_DIR/results/{name}_test.csv

In [ ]:
for name in RUNS:
    print(f"=== {name} ===")
    !python scripts/evaluate.py --generations $DATA_DIR/results/{name}_test.csv
    print()